### Minicurso Sistemas Multi-agente com LangGraph

## Exemplo com Ferramentas de APIs externas

Moacir Antonelli Ponti - 2025

---

In [35]:
import requests
from typing import TypedDict, List
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import ToolNode
from dotenv import load_dotenv
from IPython.display import Image, display

load_dotenv()

True

In [22]:
# TOOLS (APIs públicas)
@tool
def clima(cidade: str) -> str:
    """Consulta clima de cidades por API pública."""
    url = f"https://wttr.in/{cidade}?format=j1"
    r = requests.get(url)
    data = r.json()

    current = data["current_condition"][0]
    temp = current["temp_C"]
    desc = current["weatherDesc"][0]["value"]
    return f"{cidade}: {temp}°C, {desc}"

@tool
def cambio_usd_brl() -> str:
    """Cotação USD BRL via API pública."""
    url = "https://economia.awesomeapi.com.br/json/last/USD-BRL"
    r = requests.get(url).json()
    valor = float(r["USDBRL"]["bid"])
    return f"1 USD = {valor:.2f} BRL"


In [23]:
class AgentState(TypedDict):
    messages: List

tools = [clima, cambio_usd_brl]

llm = ChatOpenAI(
    temperature=0,
    model="gpt-4o-mini"
).bind_tools(tools)

In [ ]:
def agente_llm(state: AgentState) -> AgentState:
    """Nó principal: o LLM decide se responde ou chama tool."""
    msg = llm.invoke(state["messages"])
    return {"messages": state["messages"] + [msg]}

def executor_tool(state: AgentState) -> AgentState:
    """Se última mensagem do LLM for uma chamada de ferramenta, executa a ferramenta."""
    last = state["messages"][-1]

    if not hasattr(last, "tool_calls") or len(last.tool_calls) == 0:
        return state  # Nada a executar

    for call in last.tool_calls:
        tool_name = call["name"]
        args = call["args"]

        if tool_name == "clima":
            result = clima.invoke(args)
        elif tool_name == "cambio_usd_brl":
            result = cambio_usd_brl.invoke(args)
        else:
            result = f"Tool desconhecida: {tool_name}"

        tool_msg = ToolMessage(
            content=str(result),
            tool_call_id=call["id"]
        )

        return {"messages": state["messages"] + [tool_msg]}

In [32]:
graph = StateGraph(AgentState)

graph.add_node("AgenteLLM", agente_llm)
graph.add_node("ExecutorTool", executor_tool)

graph.set_entry_point("AgenteLLM")

# Roteamento condicional: se o LLM chamar uma tool: executa; senão: termina
def roteamento_tool(state: AgentState):
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and len(last.tool_calls) > 0:
        return "ExecutorTool"
    return END

graph.add_conditional_edges("AgenteLLM", roteamento_tool)

# Após executar a tool, volta para o LLM (feedback loop)
graph.add_edge("ExecutorTool", "AgenteLLM")

app = graph.compile()

In [34]:
# Clima
out = app.invoke({"messages": [HumanMessage(content="Como está o clima em São Carlos?")]})
print(out["messages"][-1].content)

# Câmbio
out = app.invoke({"messages": [HumanMessage(content="Quanto está o dólar agora?")]})
print(out["messages"][-1].content)

O clima em São Carlos está a 22°C, com chuvas leves e neblina.
Atualmente, 1 dólar americano (USD) está cotado a 5,30 reais (BRL).
